# Watermark Segmentation - Colab Training

Trains a single-class YOLO11-seg model to segment the **AriaTender** watermark.

**Upload one file:** `wm_dataset_colab.zip` (78 MB - 588 images + 588 YOLO-seg labels;
500 train / 88 val, of which 80 are negatives).

At 78 MB the browser upload in section 3 is slow and occasionally drops the connection.
**Google Drive is the more reliable route** - that cell is no longer commented out.

**Bring back:** `best.pt`.

---
### Why the settings differ from the local run

Locally this ran on an MX150 (~2 GB VRAM) at ~72 s/iteration - about 60 hours for 60 epochs,
while spilling past VRAM into shared memory. A Colab T4/L4 removes that constraint, so this
notebook raises `imgsz` back to **1024**. That matters specifically here: the watermark contains
a thin-ring shield logo whose strokes get destroyed at low resolution, and that is the exact
shape the current SAM-based detector already fails on. Resolution is the main lever for it.

### What is in this dataset

**Two watermarks**, both single-class (`watermark`), roughly evenly split:

- `ariatender_stacked` - the logo with the Persian subtitle beneath it, flat grey.
- `ariatender_wide` - the wide `AriaTender.neT` wordmark with the pink shield/gavel. This one is
  two-tone, so the model sees chromatic watermark pixels as well as neutral ones.

Both are composited at a *calibrated* semi-transparency (alpha 0.3125 against a fixed grey ink),
not pasted opaquely - the document underneath stays visible through the mark, which is the whole
property the model has to learn.

**Nine background archetypes**, so the mark is not always sitting on white paper: dense Persian
body text, tender notices, letters, tables, forms, spreadsheets, near-empty pages, pages carrying
photographic/chart imagery, and landscape slides. Roughly 18% of pages use a **dark** ground and
another ~20% a strongly tinted coloured one. On a dark page the semi-transparent mark composites
*lighter* than its surroundings rather than darker - a genuinely different appearance the model
would otherwise never see.

**Training data is entirely synthetic** (the watermark composited onto generated Persian
document pages). Whether it transfers to real documents is the open question - judge that on
real images after bringing `best.pt` back, not from the numbers in this notebook.


## 1. Check the GPU

If this fails: `Runtime -> Change runtime type -> T4 GPU`.


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
print('device:', torch.cuda.get_device_name(0))


## 2. Install ultralytics

Pinned to the version used locally so results stay comparable.


In [ ]:
%pip -q install ultralytics==8.4.142
import ultralytics; ultralytics.checks()


## 3. Upload the dataset

Run the cell, then choose **`wm_dataset_colab.zip`**.

If the browser upload is slow or drops, use the Google Drive cell below instead.


In [ ]:
import pathlib
from google.colab import files

ZIP = pathlib.Path('wm_dataset_colab.zip')

# Order of preference: already here (sidebar drag / previous run) > Drive > browser upload.
# The browser upload is last because this zip is ~78 MB, which files.upload() handles slowly
# and sometimes drops outright.
if ZIP.exists():
    print(f'{ZIP} already present ({ZIP.stat().st_size/1e6:.1f} MB), skipping upload.')
else:
    print('wm_dataset_colab.zip not found in /content.')
    print('Run the Drive cell below (preferred), or upload through the browser now.')
    uploaded = files.upload()   # choose wm_dataset_colab.zip


*Google Drive - the preferred route for this file.* Put `wm_dataset_colab.zip` in your Drive
root, then run the cell below. It is uncommented because at 78 MB the browser upload above is
the less reliable of the two.


In [ ]:
import pathlib, shutil

ZIP = pathlib.Path('/content/wm_dataset_colab.zip')
if ZIP.exists():
    print('zip already present, nothing to do')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    src = pathlib.Path('/content/drive/MyDrive/wm_dataset_colab.zip')
    assert src.exists(), f'{src} not found - put the zip in your Drive root (or edit this path).'
    shutil.copy(src, ZIP)
    print(f'copied from Drive ({ZIP.stat().st_size/1e6:.1f} MB)')


## 4. Unzip and write `data.yaml` with ABSOLUTE paths

This is deliberate, not boilerplate. Ultralytics resolves a relative `path:` against its own
`settings['datasets_dir']` - **not** the working directory - so a relative path silently looks in
the wrong place and training reads nothing. Absolute paths avoid that, and the cell then confirms
it using ultralytics' own resolver rather than assuming.


In [ ]:
import zipfile, pathlib, yaml

ROOT = pathlib.Path('/content/wm_dataset_out')
if not ROOT.exists():
    with zipfile.ZipFile('wm_dataset_colab.zip') as z:
        z.extractall('/content')

data = {
    'path':  str(ROOT),          # ABSOLUTE - see note above
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['watermark'],
}
yaml_path = pathlib.Path('/content/data.yaml')
yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print(yaml_path.read_text())

from ultralytics.data.utils import check_det_dataset
resolved = check_det_dataset(str(yaml_path))
print('resolved train ->', resolved['train'])
print('resolved val   ->', resolved['val'])


## 5. Sanity-check the dataset before spending GPU time

Confirms every image has a matching label, and counts the intentional **negatives** (empty label
files = clean pages with no watermark). Ultralytics treats those as background images, which is
wanted - they teach the model not to fire on clean documents. A negative count of 0 here means
the upload is wrong.


In [ ]:
for split in ['train', 'val']:
    imgs = sorted((ROOT/'images'/split).glob('*'))
    labs = sorted((ROOT/'labels'/split).glob('*.txt'))
    stems_i = {p.stem for p in imgs}
    stems_l = {p.stem for p in labs}
    empties = [p for p in labs if p.stat().st_size == 0]
    print(f'{split:5} images={len(imgs):4d} labels={len(labs):4d} '
          f'unmatched={len(stems_i ^ stems_l):3d} negatives={len(empties):3d}')
    assert not (stems_i ^ stems_l), f'{split}: image/label filenames do not match'


## 6. Train

201 train / 35 val images. On a T4 expect roughly 20-40 minutes.

`patience=30` early-stops if validation stops improving - the dataset is small and will overfit
quickly. On CUDA OOM: drop `batch` to 8, then 4, and only reduce `imgsz` as a last resort.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')   # COCO-pretrained, nano

results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=1024,      # thin-ring shield logo needs the resolution
    batch=16,
    patience=30,
    seed=0,
    project='/content/runs',
    name='wm_seg',
    plots=True,
)


## 7. Validate and look at predictions

These mAP numbers are on **held-out synthetic** data. They show the model learned the training
distribution; they do **not** show it works on real scans. The plotted validation batches below
are the more informative output.


In [ ]:
metrics = model.val()
print('mask mAP50    :', round(float(metrics.seg.map50), 4))
print('mask mAP50-95 :', round(float(metrics.seg.map), 4))
print('mask precision:', round(float(metrics.seg.mp), 4))
print('mask recall   :', round(float(metrics.seg.mr), 4))


In [ ]:
import glob
from IPython.display import Image as IPyImage, display

for p in sorted(glob.glob('/content/runs/wm_seg/val_batch*_pred.jpg'))[:3]:
    print(p)
    display(IPyImage(filename=p, width=900))


## 8. Inference on the REAL test set

This is the section that actually matters. Everything above is measured on synthetic data the
model was trained to fit; this is the first time it sees real documents.

**These images have no labels, so there are no accuracy numbers here and the notebook does not
invent any.** IoU / precision / recall are comparisons against ground truth and are undefined
without it. What you get instead is the overlays plus descriptive statistics. Judge by eye:

- Does it find the diagonal mark on real screenshots at all?
- Does it fire on things it obviously should not - body text, table rules, UI chrome?

A model that scores well above but finds nothing here has learned the compositing artifacts
rather than the watermark, which is the main risk of training on synthetic data.


### 8a. Upload the test folder, then set its name

Your layout: a flat folder of images, no labels.

```
folder_name/
    image1.jpg
    image2.jpg
    ...
```

**Easiest way to get it in:** zip the folder on your machine, then upload the zip below - Colab's
file picker cannot upload a directory, and dragging many files into the sidebar is slow and drops
files silently. If you already dragged the folder in, or mounted Drive, just set `TEST_DIR` and
skip the upload.


In [ ]:
# >>> SET THIS to your folder name (or a full path, e.g. '/content/drive/MyDrive/my_tests')
TEST_DIR = 'folder_name'

import pathlib, zipfile
from google.colab import files

test_path = pathlib.Path(TEST_DIR)
if not test_path.is_absolute():
    test_path = pathlib.Path('/content') / test_path

if not test_path.exists():
    print(f'{test_path} not found - upload a .zip of the folder now (or Cancel and',
          'drag the folder into the file browser, or mount Drive).')
    up = files.upload()
    for fname in up:
        if fname.lower().endswith('.zip'):
            with zipfile.ZipFile(fname) as z:
                z.extractall('/content')
            print('extracted', fname)

# If the zip contained a single top-level folder, fall through to it.
if not test_path.exists():
    cands = [p for p in pathlib.Path('/content').iterdir()
             if p.is_dir() and p.name not in {'sample_data', 'runs', 'wm_dataset_out'}]
    print('Did not find', test_path, '- candidate folders in /content:',
          [p.name for p in cands])

assert test_path.exists(), f'Set TEST_DIR to a real folder. {test_path} does not exist.'

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
test_images = sorted(p for p in test_path.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
print(f'{test_path}  ->  {len(test_images)} images')
assert test_images, f'No images found under {test_path}'


### 8b. Run inference and render overlays

Runs at the same `imgsz` used for training. Lower `CONF` to see marks the model is unsure about
(useful for judging whether it is missing them outright vs. merely scoring them low).


In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

CONF = 0.25
IMGSZ = 1024          # match training
MASK_COLOR = (255, 40, 40)
MASK_ALPHA = 0.45

best_w = pathlib.Path('/content/runs/wm_seg/weights/best.pt')
assert best_w.exists(), f'No trained weights at {best_w} - run the training cell first.'
infer_model = YOLO(str(best_w))

OUT_DIR = pathlib.Path('/content/test_overlays'); OUT_DIR.mkdir(exist_ok=True)
rows = []

for i, p in enumerate(test_images, 1):
    img = Image.open(p).convert('RGB')
    W, H = img.size
    r = infer_model.predict(source=str(p), conf=CONF, imgsz=IMGSZ, verbose=False)[0]

    masks = None
    if r.masks is not None and r.masks.data is not None and len(r.masks.data):
        m = r.masks.data.cpu().numpy()
        resized = [(np.asarray(Image.fromarray((s * 255).astype(np.uint8)).resize((W, H), Image.NEAREST)) > 127)
                   for s in m]
        masks = np.stack(resized) if resized else None
    confs = [float(c) for c in r.boxes.conf.cpu().numpy()] if (r.boxes is not None and len(r.boxes)) else []

    arr = np.asarray(img).astype(np.float32)
    if masks is not None:
        union = masks.sum(axis=0) > 0
        arr[union] = (1 - MASK_ALPHA) * arr[union] + MASK_ALPHA * np.array(MASK_COLOR, np.float32)
    overlaid = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    n_inst = 0 if masks is None else int(masks.shape[0])
    cov = 0.0 if masks is None else float((masks.sum(axis=0) > 0).mean())

    panel = Image.new('RGB', (W * 2 + 8, H + 26), (245, 245, 245))
    panel.paste(img, (0, 26)); panel.paste(overlaid, (W + 8, 26))
    d = ImageDraw.Draw(panel)
    d.text((4, 6), f'original - {p.name}', fill=(40, 40, 40))
    d.text((W + 12, 6), f'prediction: {n_inst} inst, {cov*100:.2f}% of page', fill=(180, 30, 30))
    panel.save(OUT_DIR / f'{p.stem}_overlay.png')

    rows.append({'image': p.name, 'instances': n_inst, 'coverage': cov,
                 'conf_max': max(confs) if confs else 0.0})
    if i % 20 == 0 or i == len(test_images):
        print(f'  [{i}/{len(test_images)}]')

n_any = sum(1 for r_ in rows if r_['instances'] > 0)
covs = [r_['coverage'] for r_ in rows]
inst = [r_['instances'] for r_ in rows]
print('\n' + '=' * 64)
print('DESCRIPTIVE ONLY - no ground truth, so these are NOT accuracy metrics')
print('=' * 64)
print(f'  images                    : {len(rows)}')
print(f'  images with >=1 detection : {n_any}  ({100*n_any/len(rows):.1f}%)')
print(f'  instances/image           : mean {np.mean(inst):.1f}  median {np.median(inst):.0f}  max {max(inst)}')
print(f'  page coverage             : mean {100*np.mean(covs):.2f}%  median {100*np.median(covs):.2f}%  max {100*max(covs):.2f}%')


### 8c. Look at the overlays

Red = predicted watermark pixels. Shows the first 12; all of them are saved to
`/content/test_overlays/`.


In [ ]:
from IPython.display import Image as IPyImage, display

SHOW = 12
for r_ in rows[:SHOW]:
    stem = pathlib.Path(r_['image']).stem
    print(f"{r_['image']}  ->  {r_['instances']} inst, {r_['coverage']*100:.2f}% coverage, "
          f"max conf {r_['conf_max']:.2f}")
    display(IPyImage(filename=str(OUT_DIR / f'{stem}_overlay.png'), width=1100))


*Optional - download every overlay as a zip to review offline.*


In [ ]:
import shutil
shutil.make_archive('/content/test_overlays', 'zip', str(OUT_DIR))
from google.colab import files
files.download('/content/test_overlays.zip')


## 9. Download `best.pt`

This is the file to send back.


In [ ]:
import pathlib
from google.colab import files

best = pathlib.Path('/content/runs/wm_seg/weights/best.pt')
print('exists:', best.exists(), '| size:', round(best.stat().st_size / 1e6, 2), 'MB')
files.download(str(best))


*Optional - also copy the whole run to Drive, in case the session disconnects.*


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/runs/wm_seg '/content/drive/MyDrive/wm_seg_run'
